In [1]:
import random
import pickle
import numpy as np
from collections import defaultdict
import scipy.sparse as sp

import os
import random
import pandas as pd
import json
import pickle
import gzip
from tqdm import tqdm

def load_pickle(filename):
    with open(filename, "rb") as f:
        return pickle.load(f)


def save_pickle(data, filename):
    with open(filename, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

def load_json(file_path):
    with open(file_path, "r") as f:
        return json.load(f)
    
def ReadLineFromFile(path):
    lines = []
    with open(path,'r') as fd:
        for line in fd:
            lines.append(line.rstrip('\n'))
    return lines

def parse(path):
    g = gzip.open(path, 'r')
    for l in g:
        # Convert bytes to string
        data_str = l.decode('utf-8')
        # Replace 'false' with 'False' and 'true' with 'True'
        data_str = data_str.replace('false', 'False').replace('true', 'True')

        # Parse the JSON string into a dictionary
        yield eval(data_str)
'''
Set seeds
'''
seed = 999
random.seed(seed)
np.random.seed(seed)

In [2]:
DATA_PATH = 'data/'

In [3]:
DATASET = 'yelp'

In [4]:
class DataCreator():
    def __init__(self):
        
        rev_data = load_pickle(os.path.join(DATA_PATH,DATASET,"review_splits.pkl"))
        print("Rev data keys: ",rev_data.keys())
        
        datamaps = load_json(os.path.join(DATA_PATH,DATASET,"datamaps.json"))
        print("Datamaps keys: ",datamaps.keys())
        
        self.training_data =  rev_data['train']
        self.val_data = rev_data['val']
        self.test_data = rev_data['test']
        
        self.user = datamaps['user2id']
        self.item = datamaps['item2id']
        self.id2user = datamaps['id2user']
        self.id2item = datamaps['id2item']
                
        self.__get_interact_data() # ADDED for convenience; changes the training data variable
                
        self.user_num = len(self.id2user)
        self.item_num = len(self.id2item)
        
        print("User Count:" ,self.user_num)
        print("Item Count:",self.item_num)
                
        
    def __get_interact_data(self):
        train_inter = []
        for pair in self.training_data:
            user,item,rating = pair['reviewerID'],pair['asin'],pair['overall']
            train_inter.append((user,item,rating))
        
        val_inter = []
        for pair in self.val_data:
            user,item,rating = pair['reviewerID'],pair['asin'],pair['overall']
            val_inter.append((user,item,rating))
        
        test_inter = []
        for pair in self.test_data:
            user,item,rating = pair['reviewerID'],pair['asin'],pair['overall']
            test_inter.append((user,item,rating))
            
        self.training_data = train_inter
        self.val_data = val_inter
        self.test_data = test_inter


    def __create_sparse_interaction_matrix(self):
        """
        return a sparse adjacency matrix with the shape (user number, item number)
        """
        row, col, entries = [], [], []
        for pair in self.training_data:
            row += [int(self.user[pair[0]]) - 1] # Correction July 22: apply correction (minus 1) for the index!
            col += [int(self.item[pair[1]]) - 1] # Correction July 22: apply correction (minus 1) for the index!
            entries += [1.0]
        
        interaction_mat = sp.csr_matrix((entries, (row, col)), shape=(self.user_num,self.item_num),dtype=np.float32)
        return interaction_mat


    def matrix(self):

        m = self.__create_sparse_interaction_matrix()
        return m

In [5]:
def targetItemSelect(data, popularThreshold=0.1):
    interact = data.matrix()
    userNum = interact.shape[0]
    itemNum = interact.shape[1]
    print("Data shape: ",interact.shape)
    targetNum = int(popularThreshold * itemNum)
    
    path = os.path.join(DATA_PATH,DATASET,"targetItems.txt")
    if os.path.exists(path):
        with open(path, 'r') as f:
            line = f.read()
            targetItem = [i.replace("'", "") for i in line.split(",")]
        print("Loaded target items from ",path)
        print("# target items: ",len(targetItem))
        return targetItem
    else:
        def getPopularItemId(n):
            """
            Get the id of the top n popular items based on the number of ratings
            :return: id list
            """
            return np.argsort(interact[:, :].sum(0))[0, -n:].tolist()[0]

        
        targetItem = random.sample(set(getPopularItemId(int(popularThreshold * itemNum))),targetNum)
        
        targetItem = [data.id2item[str(i + 1)] for i in targetItem]
        with open(path, 'w') as f:
            f.writelines(str(targetItem).replace('[', '').replace(']', ''))
        
        print("# target items: ",len(targetItem))
        print("Saved target items to ",path)
        return targetItem

In [6]:
THRESH = 0.1

In [7]:
items = targetItemSelect(DataCreator(),popularThreshold=THRESH)

Rev data keys:  dict_keys(['train', 'val', 'test', 'train_indices', 'val_indices', 'test_indices'])
Datamaps keys:  dict_keys(['user2id', 'item2id', 'id2user', 'id2item', 'attribute2id', 'id2attribute'])
User Count: 30431
Item Count: 20033
Data shape:  (30431, 20033)
# target items:  2003
Saved target items to  data/yelp/targetItems.txt
